# Генерация: подготовка данных

**Рабочая выборка:** по 15 000 документов на корпус. Обрабатывается на всех трех условиях, по ней обучаются тематические модели.

**Референс:** по 20 000 документов на корпус, не пересекается с рабочей выборкой. Не обрабатывается: когерентность всех условий считается по исходным текстам.

**Тестовая партия:** первые 100 документов рабочей выборки в порядке обработки. На ней проверяется прогон перед основным запуском.

Документы калибровочных партий исключаются из обеих выборок: на отладочной партии подбирались инструкции.

In [1]:
import re
import json
import hashlib
import shutil
from datetime import datetime

import numpy as np
import pandas as pd

from src.paths import PROCESSED, DATA, PROMPTS

CORPORA = {
    "yt": PROCESSED / "yt-comments.parquet",
    "tg": PROCESSED / "tg-groups.parquet",
    "rr": PROCESSED / "ru-reviews.parquet",
}
CALIBRATION = DATA / "calibration"
GEN = DATA / "generation"
GEN.mkdir(parents=True, exist_ok=True)

LEVELS = ["level1", "level2", "level3"]
N_WORK = 15_000 
N_REF = 20_000   
N_TEST = 100     
SEED = 42

WORD = re.compile(r"[А-Яа-яЁё]+")


def n_words(text):
    return len(WORD.findall(str(text)))


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

## Корпуса

Идентификатор документа `uid`: корпус и номер документа в корпусе.

In [2]:
frames = []
for name, path in CORPORA.items():
    d = pd.read_parquet(path)
    assert {"id", "text"} <= set(d.columns), f"{name}: нет колонок id и text"
    d = d[["id", "text"]].copy()
    d.insert(0, "corpus", name)
    d.insert(0, "uid", name + "-" + d["id"].astype(str))
    frames.append(d)

corpora = pd.concat(frames, ignore_index=True)
assert corpora.uid.is_unique, "uid повторяются"
corpora["n_words"] = corpora.text.map(n_words)

print(corpora.groupby("corpus").size().rename("документов").to_string())

corpus
rr     69976
tg    298738
yt     37537


## Исключение калибровочных документов

In [3]:
calibration_uids = set()
for party in ["debug", "acceptance"]:
    path = CALIBRATION / f"{party}.parquet"
    uids = set(pd.read_parquet(path).uid)
    calibration_uids |= uids
    print(f"{party}: {len(uids)}")

found = corpora.uid.isin(calibration_uids)
print(f"найдено в корпусах: {found.sum()} из {len(calibration_uids)}")
assert found.sum() == len(calibration_uids), "часть калибровочных uid не найдена"

pool = corpora[~found]

debug: 102
acceptance: 200
найдено в корпусах: 302 из 302


## Проверка объема

In [4]:
rows = []
for name, d in pool.groupby("corpus"):
    rows.append({
        "корпус": name,
        "после фильтров": int((corpora.corpus == name).sum()),
        "калибровочных": int((corpora.corpus == name).sum() - len(d)),
        "доступно": len(d),
        "нужно": N_WORK + N_REF,
        "запас": len(d) - N_WORK - N_REF
    })
volume = pd.DataFrame(rows)
print(volume.to_string(index=False))
assert (volume["запас"] >= 0).all(), "не хватает документов"


корпус  после фильтров  калибровочных  доступно  нужно  запас
    rr           69976             85     69891  35000  34891
    tg          298738            135    298603  35000 263603
    yt           37537             82     37455  35000   2455


## Отбор

Корпус перемешивается один раз с фиксированным `seed`: первые `N_WORK` документов идут в рабочую выборку, следующие `N_REF` в референс.

In [5]:
work_parts, ref_parts = [], []
for name, d in pool.groupby("corpus"):
    shuffled = d.sample(frac=1, random_state=SEED)
    work_parts.append(shuffled.iloc[:N_WORK])
    ref_parts.append(shuffled.iloc[N_WORK:N_WORK + N_REF])

work = pd.concat(work_parts)
reference = pd.concat(ref_parts)

# общий порядок обработки: корпуса перемешаны между собой
work = work.sample(frac=1, random_state=SEED).reset_index(drop=True)
work.insert(0, "order", range(1, len(work) + 1))
reference = reference.reset_index(drop=True)

test = work.head(N_TEST).copy()

assert not set(work.uid) & set(reference.uid)
assert not set(work.uid) & calibration_uids
assert not set(reference.uid) & calibration_uids

print(f"рабочая: {len(work)}, референс: {len(reference)}, тест: {len(test)}")
print("\nтест по корпусам:")
print(test.corpus.value_counts().to_string())

рабочая: 45000, референс: 60000, тест: 100

тест по корпусам:
corpus
rr    38
yt    32
tg    30


## Длина документов

Рабочая выборка и референс должны совпадать по распределению длины с корпусом: оба отобраны случайно.

In [6]:
def length_stats(d):
    nw = d.n_words
    return {"документов": len(d), "медиана": int(nw.median()), "среднее": round(nw.mean(), 1),
            "25%": int(nw.quantile(0.25)), "75%": int(nw.quantile(0.75)), "90%": int(nw.quantile(0.9))}

rows = []
for name in CORPORA:
    for label, d in [("корпус", pool), ("рабочая", work), ("референс", reference), ("тест", test)]:
        rows.append({"корпус": name, "набор": label, **length_stats(d[d.corpus == name])})
print(pd.DataFrame(rows).to_string(index=False))

корпус    набор  документов  медиана  среднее  25%  75%  90%
    yt   корпус       37455       11     13.1    6   18   26
    yt  рабочая       15000       11     13.1    6   18   26
    yt референс       20000       11     13.1    6   18   26
    yt     тест          32        8     12.3    5   17   25
    tg   корпус      298603        8      9.6    5   12   19
    tg  рабочая       15000        8      9.6    5   12   19
    tg референс       20000        8      9.6    5   12   19
    tg     тест          30        7      8.7    6   12   15
    rr   корпус       69891       13     14.5    8   20   27
    rr  рабочая       15000       13     14.5    8   20   27
    rr референс       20000       13     14.4    8   20   27
    rr     тест          38       11     15.0    7   23   27


## Фиксация инструкций

In [7]:
snapshot = GEN / "prompts"
snapshot.mkdir(exist_ok=True)

prompts = {}
for name in LEVELS + ["batch_rule"]:
    src = PROMPTS / f"{name}.txt"
    dst = snapshot / f"{name}.txt"
    shutil.copyfile(src, dst)
    prompts[name] = sha256(dst)
    print(f"{name}: {prompts[name][:16]}")

try:
    from src.prompts import load_prompt
    rule = (snapshot / "batch_rule.txt").read_text(encoding="utf-8").strip()
    for level in LEVELS:
        loaded = load_prompt(level).strip()
        own = (snapshot / f"{level}.txt").read_text(encoding="utf-8").strip()
        print(f"{level}: load_prompt совпадает с файлом: {loaded == own}, "
              f"содержит правило формата: {rule[:30] in loaded}")
except ImportError:
    print("load_prompt не найден, сверка пропущена")

level1: 59650bfe89577d03
level2: dc268f5400131ed8
level3: 0fd59ea934e74ffe
batch_rule: f340dcf05e15b696
level1: load_prompt совпадает с файлом: True, содержит правило формата: False
level2: load_prompt совпадает с файлом: True, содержит правило формата: False
level3: load_prompt совпадает с файлом: True, содержит правило формата: False


## Сохранение

In [12]:
cols = ["uid", "corpus", "id", "text", "n_words"]
work[["order"] + cols].to_parquet(GEN / "work.parquet", index=False)
reference[cols].to_parquet(GEN / "reference.parquet", index=False)
test[["order"] + cols].to_parquet(GEN / "test.parquet", index=False)

manifest = {
    "created": datetime.now().isoformat(timespec="seconds"),
    "seed": SEED,
    "n_work": N_WORK, "n_ref": N_REF, "n_test": N_TEST,
    "sources": {name: {"file": str(path.name), "sha256": sha256(path)} for name, path in CORPORA.items()},
    "excluded_calibration": len(calibration_uids),
    "volume": volume.to_dict(orient="records"),
    "prompts_sha256": prompts,
    "files": {f: sha256(GEN / f) for f in ["work.parquet", "reference.parquet", "test.parquet"]}
}
(GEN / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

for file in ["work.parquet", "reference.parquet", "test.parquet", "manifest.json"]:
    print(f"{file}: done")

work.parquet: done
reference.parquet: done
test.parquet: done
manifest.json: done
